In [1]:
import numpy as np
import pandas as pd
df=pd.read_csv('covid_toy.csv')

In [2]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [3]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

<p>As "fever" has null values we have to perform SimpleImputer to fix it.

For "gender","city" we need to perform ohe and for "cough" we need to perform ordinal encoding.</p>

In [4]:
from sklearn.model_selection import train_test_split
x=df.drop(['has_covid'],axis=1)
y=df['has_covid']
x_train,x_test,y_train,y_test=train_test_split(x,
                                               y,
                                               test_size=0.2)

In [5]:
x_train

,age,gender,fever,cough,city
0,60,Male,103.0,Mild,Kolkata
48,66,Male,99.0,Strong,Bangalore
42,27,Male,100.0,Mild,Delhi
44,20,Male,102.0,Strong,Delhi
30,15,Male,101.0,Mild,Delhi
...,...,...,...,...,...
16,69,Female,103.0,Mild,Kolkata
93,27,Male,100.0,Mild,Kolkata
39,50,Female,103.0,Mild,Kolkata
80,14,Female,99.0,Mild,Mumbai


<h1>WITHOUT ColumnTransformer</h1>

In [6]:
#adding simple imputer to fever cell
from sklearn.impute import SimpleImputer
si=SimpleImputer()
x_train_fever=si.fit_transform(x_train[['fever']])
x_test_fever=si.transform(x_test[['fever']])

In [7]:
x_train_fever.shape

(80, 1)

In [8]:
#adding ordinal encoding to cough
from sklearn.preprocessing import OrdinalEncoder
oe=OrdinalEncoder(categories=[['Mild','Strong']])
x_train_cough=oe.fit_transform(x_train[['cough']])
x_test_cough=oe.transform(x_test[['cough']])

In [9]:
x_train_cough.shape

(80, 1)

In [10]:
#adding ohe to gender and city
from sklearn.preprocessing import OneHotEncoder
ohe=OneHotEncoder(drop='first',sparse_output=False)
x_train_gender_city=ohe.fit_transform(x_train[['gender','city']])
x_test_gender_city=ohe.transform(x_test[['gender','city']])

In [11]:
x_train_gender_city.shape

(80, 4)

In [12]:
#extracting age
x_train_age=x_train.drop(['gender','fever','cough','city'],axis=1).values
x_test_age=x_test.drop(['gender','fever','cough','city'],axis=1).values

In [13]:
x_train_age.shape

(80, 1)

In [14]:
#joining
x_train_transformed=np.concatenate((x_train_age,x_train_fever,
                                    x_train_gender_city,x_train_cough),
                                   axis=1)

x_test_transformed = np.concatenate((x_test_age,x_test_fever,
                                     x_test_gender_city,x_test_cough),
                                    axis=1)

In [15]:
x_train_transformed.shape

(80, 7)

In [16]:
x_train_transformed

array([[ 60.        , 103.        ,   1.        ,   0.        ,
          1.        ,   0.        ,   0.        ],
       [ 66.        ,  99.        ,   1.        ,   0.        ,
          0.        ,   0.        ,   1.        ],
       [ 27.        , 100.        ,   1.        ,   1.        ,
          0.        ,   0.        ,   0.        ],
       [ 20.        , 102.        ,   1.        ,   1.        ,
          0.        ,   0.        ,   1.        ],
       [ 15.        , 101.        ,   1.        ,   1.        ,
          0.        ,   0.        ,   0.        ],
       [ 51.        , 101.        ,   0.        ,   0.        ,
          1.        ,   0.        ,   1.        ],
       [ 64.        ,  98.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ],
       [ 59.        ,  99.        ,   0.        ,   1.        ,
          0.        ,   0.        ,   1.        ],
       [ 24.        , 102.        ,   0.        ,   0.        ,
          0.    

<h1>WITH COLUMN TRANSFORMER</h1>

In [17]:
from sklearn.compose import ColumnTransformer

In [18]:
transformer=ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(drop='first',sparse_output=False),['gender','city'])
    ],remainder='passthrough')

In [19]:
x_train_final=transformer.fit_transform(x_train)
x_test_final=transformer.transform(x_test)

In [20]:
x_train_final.shape

(80, 7)

<p>ColumnTransformer is used when different columns of a dataset require different preprocessing techniques. Instead of preprocessing every column separately and then manually combining them, ColumnTransformer allows all transformations to be performed together in a single organized pipeline. In this code, three different transformers were applied on different columns of the COVID dataset.

First, ('tnf1', SimpleImputer(), ['fever']) applies SimpleImputer on the fever column. The fever column contains missing numerical values, so SimpleImputer fills those missing values. Since no strategy was specified, the default strategy='mean' is used, meaning missing fever values are replaced by the average fever value of the column.

Second, ('tnf2', OrdinalEncoder(categories=[['Mild','Strong']]), ['cough']) applies Ordinal Encoding on the cough column. The cough column is ordinal categorical data because Mild and Strong have a meaningful order. OrdinalEncoder converts these categories into numerical values where Mild becomes 0 and Strong becomes 1 according to the order specified inside categories.

Third, ('tnf3', OneHotEncoder(drop='first', sparse_output=False), ['gender','city']) applies One Hot Encoding on the gender and city columns. These columns are nominal categorical data because their categories have no order. OneHotEncoder converts each category into separate binary columns containing 0 and 1 values. The parameter drop='first' removes one encoded column from each feature to prevent Dummy Variable Trap and multicollinearity. The parameter sparse_output=False ensures that the output is returned as a normal NumPy array instead of a sparse matrix.

Finally, remainder='passthrough' means that any column not mentioned inside the transformers list should be passed unchanged into the final dataset. After applying fit_transform(x_train), all preprocessing steps are executed together, and the final transformed dataset becomes completely numerical so that machine learning models can process it properly.</p>

<h1>LABEL ENCODING TARGET</h1>

In [21]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y_train_final=le.fit_transform(y_train)
y_test_final=le.transform(y_test)

In [22]:
y_train

0      No
48     No
42    Yes
44     No
30    Yes
     ... 
16    Yes
93    Yes
39     No
80    Yes
78    Yes
Name: has_covid, Length: 80, dtype: object

In [23]:
y_train_final

array([0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1,
       0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0,
       1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0,
       1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1])